# Silver: motivos

## Padronização
- `nome_motivo`: sem acento, MAIÚSCULO, sem espaços sobrando (mesma função usada em canais e chamados, pois é a chave de junção).
- `categoria`: sem acento e com a primeira letra maiúscula (Cartao, Beneficios, Atendimento, Financeiro, Cadastral).
- `criticidade`: minúsculo (alta, media, baixa).
- `id_motivo`: convertido para número.

## Regras de limpeza
1. Sem `id_motivo` numérico ou sem `nome_motivo`: a linha é removida.
2. `categoria` ou `criticidade` fora do domínio: o campo vira nulo e a linha fica.
3. ID repetido:
   - a) cópias idênticas: mantém uma só;
   - b) mesmo `id_motivo` com dados diferentes: anula todas;
   - c) mesmo `nome_motivo` em ids diferentes: anula todas (o join com chamados é pelo nome).
4. `categoria` nula: preenchida pelo critério de categorização do case (Transferência de agência = Cadastral, Contratação de cartão adicional = Cartao, Cancelamento de conta = Financeiro). As linhas preenchidas ficam marcadas em `categoria_imputada`.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "rocket_lab"
bronze_schema = f"{catalog}.vcredit_bronze"
silver_schema = f"{catalog}.vcredit_silver"
tabela_destino = f"{silver_schema}.motivos"

# Padronização
COM = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM = "aaaaaeeeeiiiiooooouuuuc"

def normaliza(c):
    x = F.lower(F.trim(F.col(c)))
    x = F.translate(x, COM, SEM)
    x = F.regexp_replace(x, r"[\s\u00A0]+", " ")
    return F.upper(F.trim(x))

# Ctegorias preenchidas pelo critério do case (chave = nome_motivo já padronizado)
categoria_por_motivo = {
    "TRANSFERENCIA DE AGENCIA": "Cadastral",
    "CONTRATACAO DE CARTAO ADICIONAL": "Cartao",
    "CANCELAMENTO DE CONTA": "Financeiro",
}

In [0]:
df = spark.table(f"{bronze_schema}.motivos")
n_bronze = df.count()

# Padronização das colunas
df = (df
    .withColumn("id_motivo", F.trim("id_motivo"))
    .withColumn("nome_motivo", normaliza("nome_motivo"))
    .withColumn("categoria", F.initcap(F.lower(normaliza("categoria"))))
    .withColumn("criticidade", F.lower(normaliza("criticidade")))          # alta/media/baixa
)

# Regra 1: id_motivo (só dígitos) e nome_motivo são obrigatórios -> remove a linha
valido = (
    F.col("id_motivo").rlike(r"^\d+$")
    & F.col("nome_motivo").isNotNull()
    & (F.col("nome_motivo") != "")
)
n_sem_chave = df.filter(~valido).count()
df = df.filter(valido)

# Regra 2: valor fora do domínio -> o campo vira nulo, a linha fica
cat_ok = F.col("categoria").isin("Cartao", "Beneficios", "Atendimento", "Financeiro", "Cadastral")
crit_ok = F.col("criticidade").isin("alta", "media", "baixa")
n_dominio = df.filter(F.col("categoria").isNotNull() & ~cat_ok).count()  # nulo de origem entra na regra 4
df = (df
    .withColumn("categoria", F.when(cat_ok, F.col("categoria")))
    .withColumn("criticidade", F.when(crit_ok, F.col("criticidade")))
)

# Regra 3a: cópias idênticas -> mantém uma só
cols_negocio = [c for c in df.columns if c not in ("ingest_timestamp", "source_file")]
n_antes = df.count()
df = df.dropDuplicates(cols_negocio)
n_identicas = n_antes - df.count()

# Regra 3b: mesmo id_motivo com dados diferentes -> anula todas
w_id = Window.partitionBy("id_motivo")
df = df.withColumn("qtd", F.count("*").over(w_id))
n_conflito_id = df.filter("qtd > 1").count()
df = df.filter("qtd = 1").drop("qtd")

# Regra 3c: mesmo nome_motivo em ids diferentes -> anula todas
w_nome = Window.partitionBy("nome_motivo")
df = df.withColumn("qtd", F.count("*").over(w_nome))
n_conflito_nome = df.filter("qtd > 1").count()
df = df.filter("qtd = 1").drop("qtd")

# Regra 4: categoria nula -> preenche pelo critério do case e marca a linha
df_map = spark.createDataFrame(list(categoria_por_motivo.items()), ["nome_motivo", "categoria_sugerida"])
df = df.join(df_map, "nome_motivo", "left")
df = (df
    .withColumn("categoria_imputada", F.col("categoria").isNull() & F.col("categoria_sugerida").isNotNull())
    .withColumn("categoria", F.coalesce("categoria", "categoria_sugerida"))
    .drop("categoria_sugerida"))
n_imputadas = df.filter("categoria_imputada").count()
n_cat_nula = df.filter(F.col("categoria").isNull()).count()

# Colocar o id_motivo como inteiro e ordenas as colunas
df = (df.withColumn("id_motivo", F.col("id_motivo").cast("int"))
        .select("id_motivo", "nome_motivo", "categoria", "criticidade",
                "categoria_imputada", "ingest_timestamp"))

# Gravação (lê da bronze e sobrescreve)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(tabela_destino))

# Resumo
n_silver = spark.table(tabela_destino).count()
print(f"bronze: {n_bronze} linhas")
print(f"  removidas por id/nome inválido:         {n_sem_chave}")
print(f"  categoria fora do domínio (anulada):    {n_dominio}")
print(f"  cópias idênticas removidas:             {n_identicas}")
print(f"  linhas anuladas por id conflitante:     {n_conflito_id}")
print(f"  linhas anuladas por nome conflitante:   {n_conflito_nome}")
print(f"  categorias preenchidas pelo case:       {n_imputadas}")
print(f"  categorias ainda nulas:                 {n_cat_nula}")
print(f"silver: {n_silver} linhas")

display(spark.table(tabela_destino).orderBy("id_motivo"))